# 1 - CONFIGURAÇÕES INICIAIS



In [206]:
# ==============================
# 1. Configurações iniciais
# ==============================

import pandas as pd
import numpy as np
from pathlib import Path

# Caminho base do projeto (ajuste para a sua estrutura)
PROJECT_ROOT = Path("/content/drive/MyDrive/PROJETOS/FINANCEIRO/ETL/DATA")

BRONZE_PATH = PROJECT_ROOT / "BRONZE"
SILVER_PATH = PROJECT_ROOT / "SILVER"

SILVER_PATH.mkdir(parents=True, exist_ok=True)

# Data de corte para cálculo de atraso (fotografia da carteira)
DATA_CORTE = pd.to_datetime("2025-12-31")

print("BRONZE_PATH:", BRONZE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("DATA_CORTE :", DATA_CORTE)


BRONZE_PATH: /content/drive/MyDrive/PROJETOS/FINANCEIRO/ETL/DATA/BRONZE
SILVER_PATH: /content/drive/MyDrive/PROJETOS/FINANCEIRO/ETL/DATA/SILVER
DATA_CORTE : 2025-12-31 00:00:00


# 2 - LEITURA DOS DADOS BRONZE


In [207]:
# ==============================
# 2. Leitura dos dados Bronze
# ==============================

# Dimensões
df_clientes = pd.read_csv(BRONZE_PATH / "clientes.csv")
df_fornecedores = pd.read_csv(BRONZE_PATH / "fornecedores.csv")
df_centros_custo = pd.read_csv(BRONZE_PATH / "centros_custo.csv")
df_plano_contas = pd.read_csv(BRONZE_PATH / "plano_contas.csv")
df_calendario = pd.read_csv(BRONZE_PATH / "calendario.csv", parse_dates=["data"])

# Fatos operacionais (contas a receber/pagar)
df_contas_a_receber = pd.read_csv(
    BRONZE_PATH / "contas_receber.csv",
    parse_dates=["data_emissao", "data_vencimento", "data_pagamento"],
)

df_contas_a_pagar = pd.read_csv(
    BRONZE_PATH / "contas_pagar.csv",
    parse_dates=["data_emissao", "data_vencimento", "data_pagamento"],
)

print("clientes:", df_clientes.shape)
print("fornecedores:", df_fornecedores.shape)
print("centros_custo:", df_centros_custo.shape)
print("plano_contas:", df_plano_contas.shape)
print("calendario:", df_calendario.shape)
print("contas_a_receber:", df_contas_a_receber.shape)
print("contas_a_pagar:", df_contas_a_pagar.shape)


clientes: (50, 5)
fornecedores: (40, 5)
centros_custo: (8, 3)
plano_contas: (10, 5)
calendario: (731, 7)
contas_a_receber: (400, 12)
contas_a_pagar: (350, 12)


## inspeção de dados

In [208]:
# ==============================
# 3. Ajuste de tipos - Dimensões
# ==============================

# Clientes
df_clientes["id_cliente"] = df_clientes["id_cliente"].astype("int64")

# Fornecedores
df_fornecedores["id_fornecedor"] = df_fornecedores["id_fornecedor"].astype("int64")

# Centros de custo
df_centros_custo["id_centro_custo"] = df_centros_custo["id_centro_custo"].astype("int64")

# Plano de contas
df_plano_contas["id_plano_contas"] = df_plano_contas["id_plano_contas"].astype("int64")

# Calendário
df_calendario["id_tempo"] = df_calendario["id_tempo"].astype("int64")

for nome, df in [
    ("df_clientes", df_clientes),
    ("df_fornecedores", df_fornecedores),
    ("df_centros_custo", df_centros_custo),
    ("df_plano_contas", df_plano_contas),
    ("df_calendario", df_calendario),
]:
    print(f"\n{nome}")
    print(df.dtypes)



df_clientes
id_cliente       int64
nome_cliente    object
segmento        object
cidade          object
estado          object
dtype: object

df_fornecedores
id_fornecedor       int64
nome_fornecedor    object
tipo_fornecedor    object
cidade             object
estado             object
dtype: object

df_centros_custo
id_centro_custo       int64
nome_centro_custo    object
tipo                 object
dtype: object

df_plano_contas
id_plano_contas     int64
codigo_contabil    object
descricao          object
grupo              object
tipo               object
dtype: object

df_calendario
data               datetime64[ns]
id_tempo                    int64
ano                         int64
mes                         int64
nome_mes                   object
dia                         int64
nome_dia_semana            object
dtype: object


# 3 - TRATAMENTO DE CONTAS_RECEBER



In [209]:
# =========================================
# 4. Tratamento de CONTAS A RECEBER (Silver)
# =========================================

df_cr = df_contas_a_receber.copy()

# Garantir tipos numéricos
cols_float = ["valor_original", "valor_pago", "juros_multa", "desconto"]
for col in cols_float:
    df_cr[col] = pd.to_numeric(df_cr[col], errors="coerce").astype("float64")

# id_centro_custo pode ter nulos -> Int64 (inteiro anulável)
df_cr["id_centro_custo"] = pd.to_numeric(df_cr["id_centro_custo"], errors="coerce").astype("Int64")
df_cr["id_plano_contas"] = df_cr["id_plano_contas"].astype("int64")
df_cr["id_cliente"] = df_cr["id_cliente"].astype("int64")

# -----------------------------
# 4.1 Dias em atraso (dias_atraso)
# -----------------------------
df_cr["dias_atraso"] = 0
df_cr["dias_atraso"] = df_cr["dias_atraso"].astype("Int64")

mask_paga = (df_cr["status"] == "PAGA") & df_cr["data_pagamento"].notna()
mask_atrasada = df_cr["status"] == "ATRASADA"

# PAGA: atraso baseado em pagamento - vencimento (só se positivo)
delta_paga = (df_cr.loc[mask_paga, "data_pagamento"] - df_cr.loc[mask_paga, "data_vencimento"]).dt.days
df_cr.loc[mask_paga, "dias_atraso"] = delta_paga.clip(lower=0).astype("Int64")

# ATRASADA: atraso baseado em DATA_CORTE - vencimento
delta_atrasada = (DATA_CORTE - df_cr.loc[mask_atrasada, "data_vencimento"]).dt.days
df_cr.loc[mask_atrasada, "dias_atraso"] = delta_atrasada.clip(lower=0).astype("Int64")

# -----------------------------
# 4.2 Valor em aberto (valor_em_aberto)
# -----------------------------
df_cr["valor_em_aberto"] = 0.0

mask_em_aberto = df_cr["status"].isin(["ABERTA", "ATRASADA"])

valor_em_aberto = (
    df_cr.loc[mask_em_aberto, "valor_original"]
    + df_cr.loc[mask_em_aberto, "juros_multa"]
    - df_cr.loc[mask_em_aberto, "desconto"]
    - df_cr.loc[mask_em_aberto, "valor_pago"]
)

df_cr.loc[mask_em_aberto, "valor_em_aberto"] = valor_em_aberto.clip(lower=0)

# -----------------------------
# 4.3 Flags de status
# -----------------------------
df_cr["flag_atrasada"] = (df_cr["status"] == "ATRASADA").astype("Int64")
df_cr["flag_em_aberto"] = df_cr["status"].isin(["ABERTA", "ATRASADA"]).astype("Int64")

print(df_cr.head())
print("\nStatus x quantidade:")
print(df_cr["status"].value_counts())
print("\nResumo valor_em_aberto:")
print(df_cr["valor_em_aberto"].describe())


   id_conta_receber  id_cliente data_emissao data_vencimento data_pagamento  \
0                 1           5   2024-11-16      2024-12-23            NaT   
1                 2          27   2025-07-09      2025-08-23            NaT   
2                 3          26   2025-11-07      2025-11-30            NaT   
3                 4          23   2025-02-08      2025-02-26     2025-02-28   
4                 5          18   2025-12-09      2026-01-04            NaT   

   id_centro_custo  id_plano_contas  valor_original  valor_pago  juros_multa  \
0                1                1        14098.68        0.00         0.00   
1                7                2         2998.22        0.00         0.00   
2                4                2        16543.85        0.00         0.00   
3                5                3        16638.81    17037.35       398.54   
4                8                2         4295.45        0.00         0.00   

   desconto     status  dias_atraso  valor_e

# 4 - TRATAMENTO DE CONTAS_PAGAR



In [210]:
# =======================================
# 5. Tratamento de CONTAS A PAGAR (Silver)
# =======================================

df_cp = df_contas_a_pagar.copy()

# Garantir tipos numéricos
cols_float = ["valor_original", "valor_pago", "juros_multa", "desconto"]
for col in cols_float:
    df_cp[col] = pd.to_numeric(df_cp[col], errors="coerce").astype("float64")

df_cp["id_centro_custo"] = pd.to_numeric(df_cp["id_centro_custo"], errors="coerce").astype("Int64")
df_cp["id_plano_contas"] = df_cp["id_plano_contas"].astype("int64")
df_cp["id_fornecedor"] = df_cp["id_fornecedor"].astype("int64")

# -----------------------------
# 5.1 Dias em atraso
# -----------------------------
df_cp["dias_atraso"] = 0
df_cp["dias_atraso"] = df_cp["dias_atraso"].astype("Int64")

mask_paga = (df_cp["status"] == "PAGA") & df_cp["data_pagamento"].notna()
mask_atrasada = df_cp["status"] == "ATRASADA"

# PAGA: histórico de atraso no pagamento
delta_paga = (df_cp.loc[mask_paga, "data_pagamento"] - df_cp.loc[mask_paga, "data_vencimento"]).dt.days
df_cp.loc[mask_paga, "dias_atraso"] = delta_paga.clip(lower=0).astype("Int64")

# ATRASADA: atraso em relação à DATA_CORTE
delta_atrasada = (DATA_CORTE - df_cp.loc[mask_atrasada, "data_vencimento"]).dt.days
df_cp.loc[mask_atrasada, "dias_atraso"] = delta_atrasada.clip(lower=0).astype("Int64")

# -----------------------------
# 5.2 Valor em aberto
# -----------------------------
df_cp["valor_em_aberto"] = 0.0

mask_em_aberto = df_cp["status"].isin(["ABERTA", "ATRASADA"])

valor_em_aberto = (
    df_cp.loc[mask_em_aberto, "valor_original"]
    + df_cp.loc[mask_em_aberto, "juros_multa"]
    - df_cp.loc[mask_em_aberto, "desconto"]
    - df_cp.loc[mask_em_aberto, "valor_pago"]
)

df_cp.loc[mask_em_aberto, "valor_em_aberto"] = valor_em_aberto.clip(lower=0)

# -----------------------------
# 5.3 Flags de status
# -----------------------------
df_cp["flag_atrasada"] = (df_cp["status"] == "ATRASADA").astype("Int64")
df_cp["flag_em_aberto"] = df_cp["status"].isin(["ABERTA", "ATRASADA"]).astype("Int64")

print(df_cp.head())
print("\nStatus x quantidade:")
print(df_cp["status"].value_counts())
print("\nResumo valor_em_aberto:")
print(df_cp["valor_em_aberto"].describe())


   id_conta_pagar  id_fornecedor data_emissao data_vencimento data_pagamento  \
0               1             28   2024-07-11      2024-08-06     2024-08-17   
1               2             28   2024-07-02      2024-07-14     2024-08-07   
2               3              6   2025-05-21      2025-06-20     2025-07-15   
3               4             12   2025-12-21      2026-01-14     2025-12-31   
4               5             24   2025-07-05      2025-07-31     2025-08-09   

   id_centro_custo  id_plano_contas  valor_original  valor_pago  juros_multa  \
0                4               12         7965.22     8192.14       226.92   
1                7               14         2504.14     2589.49        85.35   
2                1               10         2513.83     2539.83        26.00   
3                1               15        11296.64    11296.64         0.00   
4                8               11         9049.28     9498.57       449.29   

   desconto status  dias_atraso  valor

# 5 - TRATAMENTO DE DIMENSÕES



In [211]:
# =======================================
# 6. Checks de integridade referencial
# =======================================

# Contas a receber x Clientes
mask_cli_inexistente = ~df_cr["id_cliente"].isin(df_clientes["id_cliente"])
print("Contas a receber com cliente inexistente:", mask_cli_inexistente.sum())

# Contas a pagar x Fornecedores
mask_forn_inexistente = ~df_cp["id_fornecedor"].isin(df_fornecedores["id_fornecedor"])
print("Contas a pagar com fornecedor inexistente:", mask_forn_inexistente.sum())

# Contas a receber/pagar x Plano de contas
mask_pc_cr_inexistente = ~df_cr["id_plano_contas"].isin(df_plano_contas["id_plano_contas"])
mask_pc_cp_inexistente = ~df_cp["id_plano_contas"].isin(df_plano_contas["id_plano_contas"])

print("Contas a receber com plano_contas inexistente:", mask_pc_cr_inexistente.sum())
print("Contas a pagar com plano_contas inexistente:", mask_pc_cp_inexistente.sum())

# Contas a receber/pagar x Centros de custo (onde houver centro informado)
mask_cc_cr_informado = df_cr["id_centro_custo"].notna()
mask_cc_cp_informado = df_cp["id_centro_custo"].notna()

mask_cc_cr_inexistente = mask_cc_cr_informado & ~df_cr["id_centro_custo"].isin(df_centros_custo["id_centro_custo"])
mask_cc_cp_inexistente = mask_cc_cp_informado & ~df_cp["id_centro_custo"].isin(df_centros_custo["id_centro_custo"])

print("Contas a receber com centro de custo inexistente:", mask_cc_cr_inexistente.sum())
print("Contas a pagar com centro de custo inexistente:", mask_cc_cp_inexistente.sum())


Contas a receber com cliente inexistente: 0
Contas a pagar com fornecedor inexistente: 0
Contas a receber com plano_contas inexistente: 0
Contas a pagar com plano_contas inexistente: 0
Contas a receber com centro de custo inexistente: 0
Contas a pagar com centro de custo inexistente: 0


# 6 - SALVANDO CAMADA SILVER



In [212]:
# ==============================
# 7. Salvando camada Silver
# ==============================

# Fatos tratados
df_cr.to_parquet(SILVER_PATH / "contas_receber_silver.parquet", index=False)
df_cp.to_parquet(SILVER_PATH / "contas_pagar_silver.parquet", index=False)

# Dimensões tratadas
df_clientes.to_parquet(SILVER_PATH / "clientes_silver.parquet", index=False)
df_fornecedores.to_parquet(SILVER_PATH / "fornecedores_silver.parquet", index=False)
df_centros_custo.to_parquet(SILVER_PATH / "centros_custo_silver.parquet", index=False)
df_plano_contas.to_parquet(SILVER_PATH / "plano_contas_silver.parquet", index=False)
df_calendario.to_parquet(SILVER_PATH / "calendario_silver.parquet", index=False)

print("Arquivos Silver salvos em:", SILVER_PATH)


Arquivos Silver salvos em: /content/drive/MyDrive/PROJETOS/FINANCEIRO/ETL/DATA/SILVER
